# Memory Mini Demo - Conversation History

This short warm-up explains a key Agentic AI idea before the ReAct lab: **memory is usually explicit information that we pass back into the model**. It is not magic hidden recall.

In this notebook, you will use a simple non-forensic example. A student tells the assistant their name, city, date, and preference. Then we ask follow-up questions to see what the model can and cannot remember.


## Learning Goal

By the end of this mini demo, you should be able to explain three practical memory patterns:

1. `No memory`: the model only sees the current prompt.
2. `Short-term memory`: the model sees the current prompt plus earlier conversation messages.
3. `Summarized memory`: the model sees a compact list of important facts instead of the full conversation.

The most important idea: if a fact is not in the current messages or memory notes, the model should not pretend to know it.


## Memory Diagram

The diagram below shows the core idea. `chat_history` stores earlier messages. When we send that history together with the current question, the LLM can use it as short-term memory. After the LLM responds, we append the new response back to `chat_history` for the next turn.

<img src="./figures/memory_short_term_workflow.svg" alt="Diagram showing chat history and current question being sent to the LLM, then the assistant response being appended back to chat history" width="900"/>


### Step 1: Set Up the Notebook

This setup cell loads the Lab 3 model settings and creates the model client. You do not need to memorize the setup code. The important part is that later cells can send message lists to the model.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# This setup cell assumes you opened the notebook from the Lab 3 folder.
LAB_NAME = 'lab3_react_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(f'Open this notebook from the {LAB_NAME} folder.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError('Expected .env in this folder. Copy .env.example to .env first.')

load_dotenv(env_path, override=True)

MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f'MODEL or OLLAMA_BASE_URL is missing from {env_path}')

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

print('Model:', MODEL)
print('Lab folder:', lab_dir)


## Part A: No Memory

In Part A, the model receives only the follow-up question. It does **not** receive the earlier personal details, so it should say it does not know them.


### Step 2: Ask a Follow-Up Without Previous Messages

This cell intentionally leaves out the earlier context. This shows why memory matters: the model cannot remember information that was never included in the message list.


In [ ]:
# The system prompt gives the model a rule for honest memory behavior.
# If the information is not in the current messages, it should say it does not know.
memory_system_prompt = """
You are a helpful assistant teaching conversation memory.
If a fact is not present in the messages you receive, say you do not have that information.
Do not guess missing personal details.
""".strip()

# These are the details the student will provide later in the memory examples.
profile_message = (
    'My name is Maya. I live in Phoenix. The date I care about is 2026-01-03. '
    'I prefer indoor activities when the weather is hot.'
)

# This follow-up question depends on earlier details.
follow_up_question = (
    'What city am I in, what date did I mention, and what kind of activity should you suggest?'
)

# No-memory version: the model sees only the system rule and the follow-up question.
# It does not see `profile_message`, so it should not know Maya, Phoenix, or the date.
no_memory_messages = [
    {'role': 'system', 'content': memory_system_prompt},
    {'role': 'user', 'content': follow_up_question},
]

no_memory_answer = client.chat.completions.create(
    messages=no_memory_messages,
    model=MODEL,
).choices[0].message.content

print(no_memory_answer)


## Part B: Short-Term Memory With `chat_history`

Short-term memory means we keep the earlier conversation messages and send them back to the model. In this notebook, that memory is a list called `chat_history`.


### Step 3: Build a Message History

Each item in `chat_history` has two important fields:

- `role`: who said the message, such as `system`, `user`, or `assistant`
- `content`: the actual text of the message

When we include earlier messages again, the model can use them as short-term memory.

The `assistant` role means "something the model said earlier." We append assistant messages because conversation memory is not only what the user said; it is the running record of the exchange. When the full `chat_history` is sent again, the model can use those earlier turns as context.


In [ ]:
# Short-term memory starts with the system rule.
chat_history = [
    {'role': 'system', 'content': memory_system_prompt},
]

# Add the student's details to memory as a user message.
chat_history.append({'role': 'user', 'content': profile_message})

# Add a sample assistant message.
# In chat history, `assistant` means a previous model reply.
# We include it because memory usually stores both sides of the conversation.
chat_history.append({
    'role': 'assistant',
    'content': 'Thanks. I will remember those details for the next question.',
})

# Now add the follow-up question. Because the old messages are still present,
# the model can answer using the information in `profile_message`.
chat_history.append({'role': 'user', 'content': follow_up_question})

short_term_memory_answer = client.chat.completions.create(
    messages=chat_history,
    model=MODEL,
).choices[0].message.content

# Save the answer too. This is how the conversation memory grows over time.
chat_history.append({'role': 'assistant', 'content': short_term_memory_answer})

{
    'message_count_in_memory': len(chat_history),
    'answer_using_chat_history': short_term_memory_answer,
}


## Part C: Summarized Memory

Sometimes a conversation becomes too long to resend every message. A common agent pattern is to keep a short memory summary instead of the full conversation.


### Step 4: Use Compact Memory Notes

This cell stores only the important facts as `memory_notes`. The model can still answer the follow-up question because the needed facts are present.


In [ ]:
# Summarized memory keeps the important facts, not every message.
memory_notes = [
    'Student name: Maya',
    'City: Phoenix',
    'Relevant date: 2026-01-03',
    'Preference: indoor activities when the weather is hot',
]

# Turn the notes into a short block of text the model can read.
memory_context = '\n'.join(f'- {note}' for note in memory_notes)

summarized_memory_messages = [
    {'role': 'system', 'content': memory_system_prompt},
    {
        'role': 'user',
        'content': f"""
Memory notes:
{memory_context}

Current question:
{follow_up_question}
""".strip(),
    },
]

summarized_memory_answer = client.chat.completions.create(
    messages=summarized_memory_messages,
    model=MODEL,
).choices[0].message.content

{
    'memory_notes': memory_notes,
    'answer_using_summarized_memory': summarized_memory_answer,
}


## Part D: Memory Limits

Memory is useful, but it is also bounded. If a fact is not in the current message list or memory notes, the model should not invent it.


### Step 5: Ask About a Missing Detail

This cell asks for a detail that was never provided. A careful model should say it does not have enough information.


In [ ]:
missing_detail_question = 'What is my favorite food?'

# The memory notes do not include favorite food.
# A good answer should acknowledge that the information is missing.
memory_limit_messages = [
    {'role': 'system', 'content': memory_system_prompt},
    {
        'role': 'user',
        'content': f"""
Memory notes:
{memory_context}

Current question:
{missing_detail_question}
""".strip(),
    },
]

memory_limit_answer = client.chat.completions.create(
    messages=memory_limit_messages,
    model=MODEL,
).choices[0].message.content

print(memory_limit_answer)


## What This Means for ReAct

In the main Lab 3 ReAct notebook, `chat_history` works like short-term memory. It stores the question, the model's tool calls, and each tool observation. That memory lets the next ReAct turn build on what already happened instead of starting over.

Keep this distinction in mind:

- `Memory` is what the agent carries across turns.
- `Tools` are actions the agent uses to gather new information.
- `Observations` are tool results that get written back into memory.
- `Evidence-bounded answers` should only use facts that appear in memory, tools, or observations.
